# SAFE-Judge: Stage 2 ? TREC DL 2019
Run cells in order on a Colab GPU. The manifest must contain all **9,260 judged pairs from 43 queries**.
Outputs include raw scores, query-level confidence intervals, risk?coverage curves, constant-label baselines,
a query-disjoint temperature-scaling comparison, a batch-consistency check, 50 failure-review rows and a report scaffold.

Checkpoints are saved to Drive every two minutes. Corpus/model caches stay on the VM.
The five-hour runtime budget includes setup; inference stops with time reserved for export and CPU analysis.
Completion depends on the allocated GPU and measured throughput. Interrupted runs resume the same full manifest.

Build `dist/pilot_code.zip` locally using `python scripts/package_pilot.py`, then select it in the upload cell.
Fill remaining resource-plan fields below; complete the 50 manual reviews and author discussion before submission.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

if "SESSION_STARTED" not in globals():
    SESSION_STARTED = time.monotonic()
RUN = "dl2019_qwen35_full_v1"
RUNTIME_HOURS = 5.0
EXPORT_RESERVE_MINUTES = 20
MAX_BATCH_SIZE = 16
CONSISTENCY_PAIRS = 100
BOOTSTRAP_SAMPLES = 10000
UPDATE_RESOURCE_PLAN = False
UPLOAD_CODE = False  # Set True only to replace the saved source archive; changed code needs a new RUN.
RESOURCE_PLAN = {
    "assessor_gpu_model": None, "assessor_gpu_hours": None,
    "teacher_model": "meta-llama/Llama-3.1-70B-Instruct (proposed)",
    "teacher_route": None, "teacher_gpu_model": None, "teacher_gpu_hours": None,
    "api_provider": None, "api_spend_cap_usd": None,
    "manual_review_hours": None, "additional_annotation_pairs": None,
    "indexing_cpu_hours": 0, "indexing_storage_gb": 0,
    "notes": "No retrieval index is needed for judged-pair inference. Enter the remaining project budget, not this pilot's consumption. Use explicit 0/none for unused resources."
}
assert "/" not in RUN and "\\" not in RUN and RUN not in ("", ".", "..")
subprocess.run(["nvidia-smi"], check=True)

## Storage and source
Authorize the Drive mount. Upload `pilot_code.zip` once; subsequent sessions reuse it.
Reconnect using the same `RUN` to restore saved predictions. Keep configuration and source unchanged while resuming.

In [ ]:
from google.colab import drive, files
import io
import json
import shutil
import signal
import sqlite3
import stat
import zipfile
from pathlib import PurePosixPath

drive.mount("/content/drive")
repo = Path("/content/SAFE-Judge").resolve()
persistent = Path("/content/drive/MyDrive/SAFE-Judge") / RUN
persistent.mkdir(parents=True, exist_ok=True)
source_zip = persistent / "pilot_code.zip"
if UPLOAD_CODE or not source_zip.exists():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select the generated pilot_code.zip only")
    payload = next(iter(uploaded.values()))
else:
    payload = source_zip.read_bytes()
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    allowed = {"src", "scripts", "tests", "configs", "requirements-pilot.txt", "README.md", "notebooks", "reports", "docs"}
    for member in archive.infolist():
        p = PurePosixPath(member.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in member.filename or not p.parts or p.parts[0] not in allowed:
            raise ValueError("Unexpected archive path")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("Archive symlinks are not supported")
    repo.mkdir(exist_ok=True)
    archive.extractall(repo)
if UPLOAD_CODE or not source_zip.exists():
    source_zip.write_bytes(payload)
del payload
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
manifest = repo / "data/pilot/dl2019.jsonl"
config_path = repo / "configs/colab.yaml"
output = repo / "results/stage2" / RUN
backup = persistent / "results/stage2" / RUN
review = output / "failure_review.csv"
for local in (manifest, manifest.with_suffix(".metadata.json"), config_path):
    saved = persistent / local.relative_to(repo)
    if not local.exists() and saved.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(saved, local)
if not (output / "checkpoint.sqlite3").exists() and (backup / "checkpoint.sqlite3").exists():
    shutil.copytree(backup, output, dirs_exist_ok=True)
output.mkdir(parents=True, exist_ok=True)
# A fresh VM has no process from a previous runtime. Same-VM active locks are preserved.
lock = output / "run.lock"
if lock.exists():
    import socket
    previous = json.loads(lock.read_text())
    if previous.get("host") != socket.gethostname():
        lock.unlink()
    else:
        raise RuntimeError("An existing same-VM lock needs inspection before another inference process starts")

def persist():
    for local in (manifest, manifest.with_suffix(".metadata.json"), config_path):
        if local.exists():
            saved = persistent / local.relative_to(repo)
            saved.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local, saved)
    for local in output.rglob("*"):
        if not local.is_file() or local.name == "run.lock" or local.name.startswith("checkpoint.sqlite3"):
            continue
        saved = persistent / local.relative_to(repo)
        if local.name == "failure_review.csv" and saved.exists():
            continue  # Drive is authoritative for human annotations.
        saved.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local, saved)
    db = output / "checkpoint.sqlite3"
    if db.exists():
        from src.utils.io import PredictionStore
        store = PredictionStore(db, json.loads((output / "experiment.json").read_text()))
        try:
            store.backup(backup / "checkpoint.sqlite3")
        finally:
            store.close()

def run(*args, allowed_codes=(0,), log_name="workflow.log"):
    command = [sys.executable, *map(str, args)]
    with (output / log_name).open("a", encoding="utf-8") as log:
        log.write("\n" + " ".join(command) + "\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log.flush()
            persist()
    if code not in allowed_codes:
        raise subprocess.CalledProcessError(code, command)
    return code
print("Saved run:", persistent)

## Environment
Use Colab's CUDA PyTorch. Other inference packages are pinned. Package versions are part of checkpoint identity.

In [ ]:
packages = ["transformers==5.3.0", "accelerate==1.15.0", "bitsandbytes==0.50.2",
            "ir-datasets==0.5.11", "PyYAML==6.0.2", "matplotlib==3.10.1"]
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
import torch
import importlib.metadata
import platform
from datetime import datetime, timezone
from src.utils.io import file_hash, read_manifest, write_json

assert torch.cuda.is_available(), "Select a GPU runtime"
version = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert version >= (2, 6), "PyTorch >=2.6 is required"
assert torch.cuda.get_device_capability()[0] >= 6, "NF4 requires compute capability >=6.0"
write_json(output / "environment.json", {
    "recorded_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version,
    "platform": platform.platform(), "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "cuda_runtime": torch.version.cuda, "source_zip_sha256": file_hash(source_zip),
    "runtime_budget_hours": RUNTIME_HOURS, "export_reserve_minutes": EXPORT_RESERVE_MINUTES,
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "accelerate", "bitsandbytes", "tokenizers", "huggingface-hub", "ir-datasets")}})
(output / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
run("-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py")
persist()

## Full pair manifest
Preparation uses the canonical collection and scans it without building a random-access index.
The initial corpus download and model download still require time and disk space.
The saved manifest and model revision are reused on subsequent sessions.

In [ ]:
import yaml
from transformers import AutoConfig

import resource
preparation_started = time.monotonic()
preparation_cpu = resource.getrusage(resource.RUSAGE_CHILDREN)
if not config_path.exists():
    config = yaml.safe_load(Path("configs/default.yaml").read_text())
    revision = AutoConfig.from_pretrained(config["assessor"]["model_name"], revision=config["assessor"].get("revision", "main"))._commit_hash
    if not revision or len(revision) != 40:
        raise ValueError("Model revision was not resolved")
    config["assessor"].update(revision=revision, compute_dtype="float16", batch_size=1)
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
if not manifest.exists():
    run("scripts/prepare_pilot.py", "--dataset", "trec_dl_2019", "--output", manifest,
        "--seed", 42, "--doc-access", "stream")
rows = read_manifest(manifest)
metadata = json.loads(manifest.with_suffix(".metadata.json").read_text())
assert len(rows) == 9260 and len({r["query_id"] for r in rows}) == 43
assert metadata["manifest_sha256"] == file_hash(manifest)
assert all(r["dataset"] == "trec_dl_2019" and r["relevance_threshold"] == 2 for r in rows)
if not (output / "preparation.json").exists():
    write_json(output / "preparation.json", {
        "seconds": time.monotonic() - preparation_started, "pairs": len(rows), "queries": 43,
        "child_cpu_seconds": resource.getrusage(resource.RUSAGE_CHILDREN).ru_utime + resource.getrusage(resource.RUSAGE_CHILDREN).ru_stime - preparation_cpu.ru_utime - preparation_cpu.ru_stime,
        "corpus_store": "sequential canonical collection scan; no retrieval/document lookup index",
        "manifest_sha256": file_hash(manifest), "configuration_sha256": file_hash(config_path)})
plan_path = output / "resource_plan.json"
if plan_path.exists() and not UPDATE_RESOURCE_PLAN:
    RESOURCE_PLAN = json.loads(plan_path.read_text())
else:
    write_json(plan_path, RESOURCE_PLAN)
persist()
print("Verified: 9,260 unique judged pairs / 43 queries")

## Inference
The model is loaded once. Batches are benchmarked on long inputs; the fastest successful size is selected.
CUDA OOM retries halve batch size without dropping pairs or changing input length.
The final-token projection avoids allocating full sequence-by-vocabulary logits.
A 100-pair batch-size-1 repeat follows the full run if time remains; this is a numerical/batching check, not sampled self-consistency.

In [ ]:
remaining_seconds = RUNTIME_HOURS * 3600 - (time.monotonic() - SESSION_STARTED) - EXPORT_RESERVE_MINUTES * 60
if remaining_seconds <= 0:
    persist()
    raise RuntimeError("No GPU time remains in the configured budget; resume this RUN in another session")
try:
    inference_code = run("scripts/run_pilot.py", "--manifest", manifest, "--config", config_path,
        "--output-dir", output, "--auto-batch", "--max-batch-size", MAX_BATCH_SIZE,
        "--max-seconds", remaining_seconds, "--backup-dir", backup,
        "--backup-every-seconds", 120, "--consistency-pairs", CONSISTENCY_PAIRS,
        allowed_codes=(0, 3), log_name="inference.log")
finally:
    persist()
progress = json.loads((output / "progress.json").read_text())
print(json.dumps(progress, indent=2))
if not progress["complete"]:
    print("Inference is incomplete. Resume this RUN to score all remaining pairs; partial metrics are diagnostic only.")

## Evaluation and report
All confidence intervals resample queries. Temperature is fitted on 10 calibration queries and evaluated on the remaining 33.
Threshold sweeps are descriptive; they do not establish conformal guarantees.
Manual review rows stay blank until you read and annotate them. Existing annotations are preserved.

In [ ]:
from src.utils.io import read_predictions
predictions = output / "predictions.csv"
if not predictions.exists():
    persist()
    raise RuntimeError("No predictions were produced; inspect inference.log")
scored, duplicates = read_predictions(predictions)
complete = len(scored) == 9260
arguments = ["scripts/evaluate_pilot.py", "--manifest", manifest, "--predictions", predictions,
             "--output-dir", output / "evaluation", "--plots", "--bootstrap-samples", BOOTSTRAP_SAMPLES]
if complete:
    arguments += ["--calibration-queries", 10]
else:
    arguments += ["--allow-partial"]
run(*arguments)
failures = sum(r["predicted_label"] != r["true_binary"] for r in scored)
if complete and failures >= 50:
    run("scripts/export_failures.py", "--manifest", manifest, "--predictions", predictions,
        "--output", review, "--count", 50, "--reuse-existing")
else:
    print("Manual failure export pending:", "incomplete inference" if not complete else "fewer than 50 model errors")
if (backup / "failure_review.csv").exists():
    shutil.copy2(backup / "failure_review.csv", review)
def cache_bytes(root):
    return sum(path.lstat().st_size for path in root.rglob("*") if path.is_file()) if root.exists() else 0
write_json(output / "storage.json", {
    "huggingface_cache_bytes": cache_bytes(Path.home() / ".cache/huggingface"),
    "ir_datasets_cache_bytes": cache_bytes(Path.home() / ".ir_datasets"),
    "artifact_bytes": cache_bytes(output), "free_disk_bytes": shutil.disk_usage(repo).free})
run("scripts/report_stage2.py", "--manifest", manifest, "--output-dir", output,
    "--review", review, "--resource-plan", output / "resource_plan.json")
from IPython.display import display, Markdown, Image
display(Markdown((output / "stage2_report.md").read_text()))
display(Image(filename=str(output / "evaluation/risk_coverage.png")))
persist()

## Manual review and export
Download `failure_review.csv`, edit it, save as CSV and replace it in the saved Drive output folder. For at least 50 unique failures, fill
`failure_category`, `notes`, `reviewer` and `reviewed=yes`; retain the IDs, labels and metadata file.
Read the query, full passage and model-visible passage. Use `reports/failure_analysis_protocol.md` for categories.

After editing, rerun the following cell to refresh the report and category counts. Complete `RESOURCE_PLAN` above
and the author discussion before submitting. These CPU operations do not need GPU time.

In [ ]:
# Use the saved review if it has been edited in Drive.
saved_review = backup / "failure_review.csv"
if saved_review.exists():
    shutil.copy2(saved_review, review)
write_json(output / "resource_plan.json", RESOURCE_PLAN)
if (output / "evaluation/summary.json").exists():
    run("scripts/report_stage2.py", "--manifest", manifest, "--output-dir", output,
        "--review", review, "--resource-plan", output / "resource_plan.json")
persist()
archive_path = Path("/content/stage2_artifacts.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in ("src", "scripts", "tests", "notebooks", "docs", "reports", "data/pilot"):
        for path in Path(directory).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".zip":
                archive.write(path, path.as_posix())
    for path in (Path("requirements-pilot.txt"), Path("configs/default.yaml"), config_path):
        archive.write(path, path.resolve().relative_to(repo).as_posix())
    for path in output.rglob("*"):
        if path.is_file() and path.name != "run.lock" and not path.name.endswith(("-wal", "-shm", "-journal")):
            archive.write(path, path.resolve().relative_to(repo).as_posix())
shutil.copy2(archive_path, persistent / archive_path.name)
print("Saved:", persistent / archive_path.name)
files.download(str(archive_path))